# Implement LDA to build a Topic Model

Notes:

I am following along with my M08 LDA with SKLearn Notes and my M08 HW as I complete this section.

## Setup

### Import Libraries

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA, LatentDirichletAllocation as LDA

In [ ]:
import plotly.express as px
import plotly.io as pio

sns.set_theme(style='white')
pio.renderers.default = 'vscode'

### Configuration

In [ ]:
# specify OHCO and bags
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

In [ ]:
# set chapter as bag
bag = "CHAPS"

In [ ]:
colors = "YlGnBu"

### Load LIB and CORPUS

In [ ]:
# load in tables
LIB = pd.read_csv('data/LIB.csv', sep='\t').set_index('book_id')
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)

## Prepare Data

### Build DOCS

SKLearn expects an F1 style corpus. So I create one from the annotated CORPUS table and keep only regular nouns.

In [ ]:
# only regular nouns
DOCS = CORPUS[CORPUS.pos.str.match(r'^NNS?$')]\
    .groupby(bags[bag]).term_str\
    .apply(lambda x: ' '.join(x))\
    .to_frame('doc_str')

# DOCS

### Build Count Matrix (DTM)

Now I use SKLearn's CountVectorizer to convert the F1 corpus of chapters into a document-term vector space of word counts.

In [ ]:
# instantiate CountVectorizer
count_engine = CountVectorizer(
    max_features=4000,
    max_df=.6, # drop terms appearing in more than 60% of chapters
    min_df=10, # drop terms appearing in fewer than 10 chapters (remove rare terms that would add noise without contributing to structure; hapax legomena etc)
    stop_words='english'
)

# fit the engine to the documents
# keep in mind that CountVectorizer outputs a sparse matrix
count_model = count_engine.fit_transform(DOCS['doc_str'])

# extract vocabulary array for use in PHI table
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'

# convert sparse matrix into dense Pandas dataframe
DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS) # convert to DataFrame to get DTM

DTM

## Fit LDA Model

In [ ]:
# set parameters
n_components = 15 # number of topics
random_state = 36
n_top_terms = 10 # changed from 5 for better interpretibility

In [ ]:
# DO NOT RERUN!!!

# instantiate LDA model
lda_engine = LDA(
    n_components = n_components,
    random_state = random_state
)

# fit LDA model to the DTM and generate document-topic distribution
topic_dist = lda_engine.fit_transform(DTM) 

# topic_dist

## Extract Results

### THETA (Document-Topic Matrix)

THETA: given this document, what is the probability distribution over topics? Each row is a document, each column is a topic, each cell is p(topic | document).

In [ ]:
# create THETA (Document-Topic Matrix)
THETA_raw = pd.DataFrame(
    topic_dist, # contains document-topic probabilities
    index = DOCS.index
)

THETA_raw.columns.name = 'topic_id'

### PHI (Topic-Word Matrix)

PHI: given this topic, what is the probability distribution over words? Each row is a topic, each column is a term, each cell is p(word | topic).

In [ ]:
# create PHI (Topic-Word Matrix)
PHI_raw = pd.DataFrame(
    lda_engine.components_, # contains raw pseudo-counts of words to objects
    columns = TERMS
)

PHI_raw.index.name = 'topic_id'
PHI_raw.columns.name = 'term_str'

## Save THETA and PHI and DOCS to CSV

In [ ]:
# save the DOCS table to csv
DOCS.to_csv('data/LDA_DOCS.csv', sep='\t', index=True)

# save the THETA table to csv
THETA_raw.to_csv('data/LDA_THETA.csv', sep='\t', index=True)

# save the PHI table to csv
PHI_raw.to_csv('data/LDA_PHI.csv', sep='\t', index=True)

## Load and Analyze

### Load THETA and PHI from CSV

In [ ]:
# after above was run intially, for reproducibility and to be safe and deterministic, now just load in THETA and PHI
THETA = pd.read_csv('data/LDA_THETA.csv', sep='\t').set_index(['book_id', 'chap_num'])
PHI = pd.read_csv('data/LDA_PHI.csv', sep='\t').set_index('topic_id')
# PHI.columns.name = 'term_str' # TODO: IS THIS NEEDED?

### TOPICS (Top Terms per Topic)

Creating a TOPICS table with top 10 words per topic (the top 5 are the first 5).

In [ ]:
TOPICS = PHI.stack().groupby('topic_id').apply(
    lambda x: ' '.join(x.sort_values(ascending=False)
    .head(n_top_terms).reset_index()['term_str'])
).to_frame('top_terms')

TOPICS['doc_weight_sum'] = THETA.sum(axis=0)
TOPICS['doc_weight_mean'] = THETA.mean(axis=0)

What I tried in an earlier iteration:

1. max_features=4000, max_df=0.9, min_df=10, n_components = 20
- have a lot of generic physical and spatial vocabulary (door, room, head, eyes, hand, face) in T0, T2, T7, T14, and T15

2. I am going to lower max_df from 0.9 to 0.6 to see if lowering the threshold will eliminate some of these more generic terms from the vocab.
- This has gotten better. I've noticed some french showing up in T5. I am going to drop to n_components = 15 now.

### Inspect Topics

In [ ]:
for topic_id, row in TOPICS.iterrows():
    print(f"T{str(topic_id).zfill(2)}: {row['top_terms']}")

In [ ]:
# inspect all 15 topics
TOPICS.sort_values('doc_weight_mean', ascending=False)

In [ ]:
# look at the top 5 topics
TOPICS.sort_values('doc_weight_mean', ascending=False).head(5)

Topic interpretations: # MUST REDO THIS NOW
- T9: Diffuse / Short Story Catch-all
    - a catch all that is dominant across short stories and spreadds over many works
- T7: Investigation / Detection Process
- T1: Adventure/Travel
- T13: Investigation / Interrogation / Rodger Ackroyd
- T8: War / Political Intrigue?

T00: inspector window sir doctor table friend evening minutes question chair
T01: friend word girl car letter train manner fellow hands window
T02: father millionaire secretary daughter train jewels girl lady sir car
T03: girl doctor table minutes friend lady bed window floor end
T04: girl mother money boy sister years love child letter father
T05: music darling love doctor suit point diamonds minutes men meeting
T06: train music compartment story love years world diamonds money race
T07: clocks oclock strychnine bed evidence alarm sound clock servants inquest
T08: death doctor years mistress friend sir money husband manner world
T09: police car friend lady wife men sir inspector ami road
T10: cabin deck evidence prisoner friend coffee strychnine paper point murder
T11: boy father lot years mother bit girl lady sort word
T12: chimneys letters business memoirs gentleman work bundle party hotel manuscript
T13: murder crime body magistrate police inspector window husband point murderer
T14: sir friend key paper coffee letter mistress window evidence crime

In [ ]:
# # add labels to TOPICS for riffs
# topic_labels = {
#     0: 'Interrogation/Investigation Scene',
#     1: 'Generic Social World',
#     2: 'Wealth and Upper Class Intrigue',
#     3: 'Domestic Interior',
#     4: 'Family Relationships',
#     5: 'Romance/Social Scene',
#     6: 'Train Journey/Travel',
#     7: 'Timed Evidence and Poison',
#     8: 'Death and Social Aftermath',
#     9

# }
# TOPICS['label'] = TOPICS.index.map(topic_labels)

### Explore Topic Distributions by Feature (Book/Sleuth/Work Type)

In [ ]:
THETAX = THETA.join(LIB[['sleuth', 'work_type', 'genre']], on='book_id').reset_index().set_index(['book_id', 'chap_num'])
THETAX['primary_genre'] = THETAX['genre'].str.split('|').str[0]

# THETAX

In [ ]:
# groupby book_id ("how much does this book talk about this topic?")
TOPIC_BOOK = THETAX.groupby('book_id').mean(numeric_only=True)
TOPIC_BOOK.T.style.background_gradient(axis=0, cmap="YlGnBu")

In [ ]:
TOPIC_BOOK[[1, 7, 8, 9, 13]].style.background_gradient(cmap='YlGnBu', axis=0)

In [ ]:
# sns.heatmap(TOPIC_BOOK, cmap="YlGnBu", cbar=None)
# plt.show()

In [ ]:
# groupby sleuth ("how much do works with this sleuth talk about this topic?")
THETAX['sleuth'] = THETAX['sleuth'].fillna('None') # deal with giants bread
TOPIC_SLEUTH = THETAX.groupby('sleuth').mean(numeric_only=True)
TOPIC_SLEUTH.T.style.background_gradient(axis=0, cmap="YlGnBu")

In [ ]:
TOPIC_SLEUTH[[1, 7, 8, 9, 13]].style.background_gradient(cmap='YlGnBu', axis=0)

In [ ]:
# sns.heatmap(TOPIC_SLEUTH, cmap="YlGnBu", cbar=None)
# plt.show()

In [ ]:
# groupby work type ("how much do works of this type talk about this topic?")
TOPIC_WORK = THETAX.groupby('work_type').mean(numeric_only=True)
TOPIC_WORK.T.style.background_gradient(axis=0, cmap="YlGnBu")

In [ ]:
# sns.heatmap(TOPIC_WORK, cmap="YlGnBu", cbar=None)
# plt.show()

In [ ]:
# groupby primary genre ("how much do works of this primary genre talk about this topic?")
TOPIC_GENRE = THETAX.groupby('primary_genre').mean(numeric_only=True)
TOPIC_GENRE.T.style.background_gradient(axis=0, cmap="YlGnBu")

In [ ]:
TOPIC_GENRE[[1, 7, 8, 9, 13]].style.background_gradient(cmap='YlGnBu', axis=0)

In [ ]:
# sns.heatmap(TOPIC_GENRE, cmap="YlGnBu", cbar=None)
# plt.show()

## LDA + PCA Visualization (4)

Apply PCA to the THETA table and plot the topics in the space opened by the first two components.

Size the points based on the mean document weight of each topic (using the THETA table).

Color the points basd on a metadata feature from the LIB table.

Provide a brief interpretation of what you see.

(INSERT IMAGE HERE)

(INSERT INTERPRETATION HERE)

In [ ]:
# instantiate PCA engine with defaults except random_state=36 and n_components=2
pca_engine = PCA(n_components = 2, random_state = 36)

In [ ]:
# fit PCA on THETA
pca_engine.fit(THETA)

In [ ]:
# build a data frame with topic positions in PC space
# pca.components_.T has shape (n_topics, 2)
topic_coords = pd.DataFrame(
    pca_engine.components_.T,
    index=THETA.columns,       # topic_ids 0-14
    columns=['PC0', 'PC1']
)

# add size by mean document weight per topic
topic_coords['mean_weight'] = THETA.mean(axis=0)

# add color by sleuth
TOPIC_SLEUTH = THETAX.groupby('sleuth').mean(numeric_only=True)
topic_coords['dominant_sleuth'] = TOPIC_SLEUTH.idxmax(axis=0)

# or by genre
TOPIC_GENRE = THETAX.groupby('primary_genre').mean(numeric_only=True)
topic_coords['dominant_genre'] = TOPIC_GENRE.idxmax(axis=0)

# add top terms for hover
topic_coords['top_terms'] = TOPICS['top_terms']

In [ ]:
# plot
px.scatter(
    topic_coords,
    x='PC0', y='PC1',
    size='mean_weight',
    color='dominant_sleuth',
    hover_name=topic_coords.index,
    hover_data=['top_terms'],
    title='LDA Topics in PCA Space'
)

In [ ]:
# plot
px.scatter(
    topic_coords,
    x='PC0', y='PC1',
    size='mean_weight',
    color='dominant_genre',
    hover_name=topic_coords.index,
    hover_data=['top_terms'],
    title='LDA Topics in PCA Space'
)

In [ ]:
topic_coords.sort_values('PC1', ascending=False)[['PC1', 'top_terms', 'dominant_genre', 'dominant_sleuth']]

In [ ]:
TOPIC_SLEUTH[9]

### Interpretation

PC0 separates topic 9 from everything else. Topic 9 is heavily associated with Tommy and Tuppence.

PC1 separates within the remaining topics and appears to separate investigative vocabulary from action/thriller vocabulary. (Detection vs. Thriller)

## Save Outputs

In [ ]:
# save the DOCS table to csv
DOCS.to_csv('data/LDA_DOCS.csv', sep='\t', index=True)

# save the THETA table to csv
THETA.to_csv('data/LDA_THETA.csv', sep='\t', index=True)

# save the PHI table to csv
PHI.to_csv('data/LDA_PHI.csv', sep='\t', index=True)

# save the TOPICS table to csv
TOPICS.to_csv('data/LDA_TOPICS.csv', sep='\t', index=True)